# 🚗 Dengeli Çoklu-Kaynak Araç Tespiti: UA-DETRAC + VisDrone2019 (YOLO11s)
### Bitirme Projesi — Sınıf Dengesizliği Çözüm Pipeline'ı (`car`, `motorcycle`, `bus`, `truck`)

Bu notebook, önceki modelde tespit edilen **sınıf dengesizliği** ve **`motorcycle` eksikliği** sorununu çözmek için:
1. **UA-DETRAC** üzerinde adaptive class-aware örnekleme (`truck` ve `bus` için yüksek yoğunluk) + `<ignored_region>` maskeleme uygular.
2. **VisDrone2019-DET** üzerinden yukarıdan trafik kamerası açısındaki **`motorcycle`**, **`truck`** ve **`bus`** örneklerini birleştirir.
3. Yalnızca nadir sınıflara (`motorcycle`, `truck`) hedefli trafik augmentation'ı uygular (`val`/`test` dokunulmaz).
4. **YOLO11s (`imgsz=800`)** modelini sınıflandırma kaybı ve nadir-nesne augmentation optimizasyonlarıyla eğitir ve sınıf bazlı metrik raporu üretir.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aydin-eng/federated-traffic-learning/blob/main/traffic_vehicle_detection_balanced_yolo11.ipynb)


## Adım 1: T4 GPU Kontrolü ve Bağımlılık Kurulumu


In [1]:
import torch

if torch.cuda.is_available():
    print(f'✅ GPU aktif: {torch.cuda.get_device_name(0)}')
    !nvidia-smi
else:
    print('⚠️ UYARI: GPU devrede DEĞİL! Runtime -> Change runtime type -> T4 GPU seçin.')

!pip install -q ultralytics pyyaml tqdm opencv-python-headless
print('✅ Kütüphaneler hazır!')


⚠️ UYARI: GPU devrede DEĞİL! Runtime -> Change runtime type -> T4 GPU seçin.
✅ Kütüphaneler hazır!


## Adım 2: GitHub Deposunu Çekme ve Google Drive'da `DETRAC-Images.zip` Bulma


In [2]:
import os
from google.colab import drive

repo_dir = '/content/federated-traffic-learning'
if not os.path.exists(repo_dir):
    !git clone https://github.com/aydin-eng/federated-traffic-learning.git /content/federated-traffic-learning
%cd /content/federated-traffic-learning
!git pull

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
else:
    print('✅ Google Drive zaten bağlı.')

drive_root = '/content/drive/MyDrive'
direct_zip = os.path.join(drive_root, 'DETRAC-Images.zip')
direct_dir = os.path.join(drive_root, 'DETRAC-Images')

DETRAC_SOURCE_ARG = None
if os.path.exists(direct_zip):
    zip_size = os.path.getsize(direct_zip)
    print(f'✅ DETRAC arşivi doğrudan bulundu: {direct_zip} ({zip_size / (1024**3):.2f} GB)')
    DETRAC_SOURCE_ARG = ('zip', direct_zip)
elif os.path.isdir(direct_dir):
    print(f'✅ DETRAC klasörü doğrudan bulundu: {direct_dir}')
    DETRAC_SOURCE_ARG = ('dir', direct_dir)
else:
    found_zips = []
    found_dirs = []
    skip_dirs = {'.venv', 'extracted_data', 'runs', '.trash', '__pycache__'}
    for root, dirs, files in os.walk(drive_root, followlinks=True):
        depth = os.path.relpath(root, drive_root).count(os.sep)
        if depth >= 2:
            dirs[:] = []
        else:
            dirs[:] = [d for d in dirs if d.lower() not in skip_dirs and not d.startswith('.')]
        for f in files:
            if f.lower().endswith('.zip') and 'detrac' in f.lower():
                full_p = os.path.join(root, f)
                try:
                    found_zips.append((full_p, os.path.getsize(full_p)))
                except OSError:
                    pass
        for d in dirs:
            if d.lower() == 'detrac-images':
                found_dirs.append(os.path.join(root, d))
    found_zips.sort(key=lambda x: x[1], reverse=True)
    if found_zips:
        images_zip_path, zip_size = found_zips[0]
        print(f'✅ DETRAC arşivi bulundu: {images_zip_path} ({zip_size / (1024**3):.2f} GB)')
        DETRAC_SOURCE_ARG = ('zip', images_zip_path)
    elif found_dirs:
        print(f'✅ DETRAC klasörü bulundu: {found_dirs[0]}')
        DETRAC_SOURCE_ARG = ('dir', found_dirs[0])
    else:
        raise FileNotFoundError("Google Drive'da DETRAC-Images.zip bulunamadı! 'Drive'a kısayol ekle' yaptığınızdan emin olun.")


Cloning into '/content/federated-traffic-learning'...
remote: Enumerating objects: 885, done.
remote: Counting objects: 100% (52/52), done.
remote: Compressing objects: 100% (46/46), done.
remote: Total 885 (delta 8), reused 45 (delta 4), pack-reused 833 (from 1)
Receiving objects: 100% (885/885), 163.91 MiB | 30.23 MiB/s, done.
Resolving deltas: 100% (94/94), done.
/content/federated-traffic-learning
Already up to date.


KeyboardInterrupt: 

## Adım 3: Dengeli UA-DETRAC Çıkarımı (`<ignored_region>` Maskeleme + Stratified Split)


In [ ]:
import os
import subprocess
import zipfile

annot_dir = '/content/DETRAC_annotations'
xml_zip = '/content/federated-traffic-learning/detrac_xml_annotations.zip'
if not os.path.exists(annot_dir):
    print('📦 XML etiketleri açılıyor...')
    with zipfile.ZipFile(xml_zip, 'r') as xz:
        xz.extractall(annot_dir)
    print('✅ XML etiketleri hazır!')

DATASET_DIR = '/content/balanced_traffic_dataset'
src_flag = '--images-zip' if DETRAC_SOURCE_ARG[0] == 'zip' else '--images-dir'

cmd = [
    'python', 'src/data/prepare_balanced_dataset.py',
    '--annot-dir', annot_dir,
    src_flag, DETRAC_SOURCE_ARG[1],
    '--out-dir', DATASET_DIR,
]
print('🚀 Çalıştırılıyor:', ' '.join(cmd))
subprocess.run(cmd, check=True)


## Adım 4: VisDrone2019-DET Füzyonu (`motorcycle`, `truck`, `bus`), Nadir Sınıf Augmentation ve Dağılım Raporu


In [ ]:
import glob
import os
import subprocess
from collections import Counter

DATASET_DIR = '/content/balanced_traffic_dataset'
APPLY_MINORITY_AUG = True

subprocess.run([
    'python', 'src/data/fuse_external_vehicles.py',
    '--dataset-dir', DATASET_DIR,
    '--cache-dir', '/content/visdrone_cache',
    '--max-train', '1800',
    '--max-val', '350',
], check=True)

if APPLY_MINORITY_AUG:
    subprocess.run([
        'python', 'src/data/augment_minority.py',
        '--dataset-dir', DATASET_DIR,
        '--rare-copies', '1',  # Daha uzun eğitim için rare-copies=2 yapılabilir
    ], check=True)

# Nihai sınıf dağılımı ve dengesizlik oranları raporu
class_names = {0: 'car', 1: 'motorcycle', 2: 'bus', 3: 'truck'}
print('\n' + '=' * 72)
print('        📊 NİHAİ DENGELİ VERİ SETİ SINIF DAĞILIMI RAPORU')
print('=' * 72)
for split in ('train', 'val', 'test'):
    imgs = glob.glob(os.path.join(DATASET_DIR, 'images', split, '*.*'))
    lbls = glob.glob(os.path.join(DATASET_DIR, 'labels', split, '*.txt'))
    counts = Counter()
    for lf in lbls:
        with open(lf, 'r', encoding='utf-8') as f:
            for ln in f:
                parts = ln.strip().split()
                if parts:
                    counts[int(float(parts[0]))] += 1
    total_b = sum(counts.values())
    print(f'📌 [{split.upper()}] {len(imgs):,} görsel | {total_b:,} toplam kutu')
    for cid in sorted(class_names):
        cnt = counts.get(cid, 0)
        pct = (cnt / total_b * 100) if total_b else 0.0
        print(f'   • {cid} ({class_names[cid]:<10}): {cnt:>7,} (%{pct:>5.1f})')
    car_n = counts.get(0, 0)
    ratios = [f'car:{class_names[c]}={car_n/counts[c]:.1f}:1' if counts.get(c, 0) else f'car:{class_names[c]}=inf' for c in (1, 2, 3)]
    print('   • Dengesizlik Oranları ->', ' | '.join(ratios))
print('=' * 72)


## Adım 5: Optimize Edilmiş YOLO11s Eğitimi (`imgsz=800`, `cls=0.8`, `mixup` + `copy_paste`)


In [ ]:
import torch
from ultralytics import YOLO

device_arg = 0 if torch.cuda.is_available() else 'cpu'
print(f'🚀 YOLO11s Dengeli Eğitim Başlatılıyor... Donanım: {device_arg}')

model = YOLO('yolo11s.pt')

results = model.train(
    data='/content/balanced_traffic_dataset/data.yaml',
    epochs=50,
    imgsz=800,
    batch=16,          # T4 GPU VRAM yetersiz kalırsa batch=8 yapın
    device=device_arg,
    cos_lr=True,
    lr0=0.005,
    cls=0.8,
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.15,
    close_mosaic=10,
    patience=15,
    save=True,
    plots=True,
    seed=42,
    project='runs/train',
    name='yolo11s_balanced_exp',
    verbose=True,
)


## Adım 6: Sınıf Bazlı Değerlendirme (`per_class_metrics.json`) ve Eski Model ile Kıyaslama


In [ ]:
import glob
import json
import os
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO

exp_dir = 'runs/train/yolo11s_balanced_exp'
if not os.path.exists(exp_dir):
    cands = sorted(glob.glob('runs/**/yolo11s_balanced_exp*', recursive=True))
    if cands:
        exp_dir = cands[-1]

best_weights = os.path.join(exp_dir, 'weights', 'best.pt')
print(f'🔍 Değerlendirilen yeni model: {best_weights}')
trained_model = YOLO(best_weights)

metrics = trained_model.val(data='/content/balanced_traffic_dataset/data.yaml', imgsz=800)
class_names = {0: 'car', 1: 'motorcycle', 2: 'bus', 3: 'truck'}

per_class = {}
print('\n' + '=' * 68)
print(f'{"Sınıf":<14} | {"Precision":>10} | {"Recall":>10} | {"mAP@50":>10} | {"mAP@50-95":>10}')
print('-' * 68)
for idx, cid in enumerate(metrics.box.ap_class_index):
    cid = int(cid)
    cname = class_names.get(cid, str(cid))
    p = float(metrics.box.p[idx])
    r = float(metrics.box.r[idx])
    map50 = float(metrics.box.ap50[idx])
    map95 = float(metrics.box.ap[idx])
    per_class[cname] = {'precision': p, 'recall': r, 'mAP50': map50, 'mAP50_95': map95}
    print(f'{cname:<14} | {p:>10.4f} | {r:>10.4f} | {map50:>10.4f} | {map95:>10.4f}')
print('-' * 68)
print(f'{"ALL (Mean)":<14} | {float(metrics.box.mp):>10.4f} | {float(metrics.box.mr):>10.4f} | {float(metrics.box.map50):>10.4f} | {float(metrics.box.map):>10.4f}')
print('=' * 68)

metrics_payload = {
    'overall': {
        'precision': float(metrics.box.mp),
        'recall': float(metrics.box.mr),
        'mAP50': float(metrics.box.map50),
        'mAP50_95': float(metrics.box.map),
    },
    'per_class': per_class,
}
with open(os.path.join(exp_dir, 'per_class_metrics.json'), 'w', encoding='utf-8') as f:
    json.dump(metrics_payload, f, indent=2)

# Eski baseline (weights/best.pt) vs Yeni Dengeli Model kıyaslaması
old_weights = '/content/federated-traffic-learning/weights/best.pt'
sample_images = []
user_photo = '/content/federated-traffic-learning/test_samples/user_test_photo.jpg'
if os.path.exists(user_photo):
    sample_images.append(('user_test_photo', user_photo))

# Val kümesinden truck ve motorcycle içeren 1'er örnek ekle
for target_cid, tag in ((3, 'val_truck_sample'), (1, 'val_motorcycle_sample')):
    for lf in sorted(glob.glob('/content/balanced_traffic_dataset/labels/val/*.txt')):
        with open(lf, 'r', encoding='utf-8') as f:
            if any(int(float(ln.split()[0])) == target_cid for ln in f if ln.strip()):
                stem = os.path.splitext(os.path.basename(lf))[0]
                cand = f'/content/balanced_traffic_dataset/images/val/{stem}.jpg'
                if os.path.exists(cand):
                    sample_images.append((tag, cand))
                    break

if os.path.exists(old_weights) and sample_images:
    old_model = YOLO(old_weights)
    for tag, img_p in sample_images:
        old_model.predict(source=img_p, conf=0.25, save=True, project='runs/compare', name=f'old_{tag}', exist_ok=True)
        trained_model.predict(source=img_p, conf=0.25, imgsz=800, save=True, project='runs/compare', name=f'new_{tag}', exist_ok=True)
        o_imgs = glob.glob(f'runs/compare/old_{tag}/*.*')
        n_imgs = glob.glob(f'runs/compare/new_{tag}/*.*')
        if o_imgs and n_imgs:
            fig, axes = plt.subplots(1, 2, figsize=(20, 9))
            axes[0].imshow(Image.open(o_imgs[0]))
            axes[0].set_title(f'Eski Baseline YOLO11n ({tag})', fontsize=14)
            axes[0].axis('off')
            axes[1].imshow(Image.open(n_imgs[0]))
            axes[1].set_title(f'Yeni Dengeli YOLO11s ({tag})', fontsize=14)
            axes[1].axis('off')
            plt.tight_layout()
            out_fig = os.path.join(exp_dir, f'compare_{tag}.jpg')
            plt.savefig(out_fig, bbox_inches='tight', dpi=150)
            plt.show()


## Adım 7: Sonuçları Paketleme, Google Drive'a Yedekleme ve İndirme 📦


In [ ]:
import glob
import os
import shutil
from google.colab import files

exp_dir = 'runs/train/yolo11s_balanced_exp'
if not os.path.exists(exp_dir):
    cands = sorted(glob.glob('runs/**/yolo11s_balanced_exp*', recursive=True))
    if cands:
        exp_dir = cands[-1]

# dataset_metadata.csv'yi de sonuç paketine kopyala (sonraki FL adımı için)
meta_src = '/content/balanced_traffic_dataset/dataset_metadata.csv'
if os.path.exists(meta_src):
    shutil.copy2(meta_src, os.path.join(exp_dir, 'dataset_metadata.csv'))

zip_base = 'yolo11s_balanced_results'
zip_path = shutil.make_archive(zip_base, 'zip', exp_dir)
print(f'📦 Paket hazır: {zip_path} ({os.path.getsize(zip_path) / (1024*1024):.1f} MB)')

drive_out_dir = '/content/drive/MyDrive/Colab Notebooks'
if os.path.exists('/content/drive/MyDrive'):
    os.makedirs(drive_out_dir, exist_ok=True)
    drive_dst = os.path.join(drive_out_dir, os.path.basename(zip_path))
    shutil.copy2(zip_path, drive_dst)
    print(f'☁️ Google Drive yedeği kaydedildi: {drive_dst}')

try:
    files.download(zip_path)
except Exception as e:
    print(f'ℹ️ VS Code Colab çekirdeğinde otomatik tarayıcı indirmesi atlandı ({e}).')
    print('👉 Sonuç paketi yukarıdaki Google Drive yoluna (Colab Notebooks/yolo11s_balanced_results.zip) kaydedildi!')

print('🎉 Tüm adımlar tamamlandı!')
